# The ReAct Agent

In this notebook, we will build a ReAct agent from scratch using LangGraph. The ReAct agent is a powerful agentic architecture that combines reasoning and acting capabilities, allowing it to interact with tools and make decisions based on its observations. 

In LangGraph, an agent is a workflow that uses an LLM to make decisions and interact with tools, given an instruction.

Keeping that in mind, we have to explore what tools are.

But first, we import the bare necessities, load the environment variables, and set up tracing with LangFuse.

In [ ]:
import os
import dotenv
import json
from typing import List, Annotated
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage, ToolMessage
from langchain_core.tools import tool, Tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langfuse.langchain import CallbackHandler

# Load environment variables from .env file.
dotenv.load_dotenv()

# Initialize the Langfuse handler
langfuse_handler = CallbackHandler()

## Defining our tools.

In LangGraph, a tools are utilities that can be called by an LLM to perform specific tasks. Tools can be anything from simple functions to complex APIs.

Our first tool will be a simple math tool that can perform basic arithmetic operations.

In [ ]:
class MathToolInput(BaseModel):
    """Input for the math tool."""
    operand1: float = Field(..., description="The first operand for the math operation.")
    operand2: float = Field(..., description="The second operand for the math operation.")
    operator: str = Field(..., description="The operator for the math operation", enumerate=["+", "-", "*", "/"])

@tool
def math_tool(input: MathToolInput) -> float:
    """Perform a math operation based on the input."""
    if input.operator == "+":
        return input.operand1 + input.operand2
    elif input.operator == "-":
        return input.operand1 - input.operand2
    elif input.operator == "*":
        return input.operand1 * input.operand2
    elif input.operator == "/":
        if input.operand2 == 0:
            raise ValueError("Cannot divide by zero.")
        return input.operand1 / input.operand2
    else:
        raise ValueError(f"Unknown operator: {input.operator}")

Note that we again use Pydantic. This time we use a Pydantic model to define the input schema for our tool.

Like all runnable components in LangGraph, tools can be called using the `invoke` method.

In [ ]:
math_tool.invoke({
    "input": MathToolInput(operand1=5, operand2=3, operator="+")
})

This is a simple tool. Its utility is limited and debatable. But the idea is essential: Tools can be anything, as long as they have a defined input and output schema. This makes them extremely flexible and powerful.

While we are at it, let us define a second, simple tool: A sorting tool that can sort a list of numbers in ascending or descending order.

In [ ]:
class SortToolInput(BaseModel):
    """Input for the sort tool."""
    numbers: List[float] = Field(..., description="A list of numbers to sort.")
    order: str = Field(..., description="The order to sort the numbers", enumerate=["asc", "desc"])

@tool
def sort_tool(input: SortToolInput) -> List[float]:
    """Sort a list of numbers in ascending or descending order."""
    if input.order == "asc":
        return sorted(input.numbers)
    elif input.order == "desc":
        return sorted(input.numbers, reverse=True)
    else:
        raise ValueError(f"Unknown order: {input.order}")

Let us invoke it.

In [ ]:
sort_tool.invoke({
    "input": SortToolInput(numbers=[5, 2, 9, 1, 5, 6], order="asc")
})

Beautiful! Using these two simple tools, we define a list and a lookup dictionary that maps tool names to tool instances. This will be useful later when we define our agent.

In [ ]:
# The tools and a mapping from tool name to tool.
tools: List[Tool] = [
    math_tool,
    sort_tool
]
tools_by_name = {tool.name: tool for tool in tools}

## Creating our agent.

We have already defined our tools. Next we set up our LLM and define our agent. Here it is good to keep in mind that in order to make good use of tools, we need an LLM that has been fine-tuned for tool use. We will use the Anthropic model because it is fairly good at using tools out of the box.

In [ ]:
# Initialize the chat model and bind the tools to it.
chat_model_string = os.environ["LANGCHAIN_CHAT_MODEL_ANTHROPIC"]
model_with_tools = init_chat_model(chat_model_string).bind_tools(tools)

Note that we have bound the tools to the LLM. This is important because it allows the LLM to know which tools are available to it.

Of course, agents are workflow. Thus we continue by defining the state, the nodes, and the edges.

In [ ]:
class ReActState(BaseModel):
    """State for the ReAct agent."""

    messages: Annotated[List[BaseMessage], add_messages] = Field(
        default_factory=list,
        description="The past steps taken by the agent."
    )

The state has no surprises. It is a storage for the messages that the LLM has seen so far. The list of messages will grow over time, especially when the agent uses multiple tools over time. This is our way of keeping track of the agent's memory.

The ReAct agent is not very complicated. It consists of only two nodes: The LLM node and the tool node. The LLM node is responsible for generating the next action based on the current state, while the tool node is responsible for executing the action and returning the result. The main idea is that the LLM can decide to use a tool at any time, and the tool will return a result that the LLM can use to make further decisions.

In [ ]:
def reason_node(state: ReActState) -> ReActState:
    """The reasoning node for the ReAct agent."""

    # Create the system message.
    system_message = SystemMessage(
        content="You are a helpful assistant that uses tools to answer questions. Do the best you can."
    )

    # Invoke the model.
    ai_message = model_with_tools.invoke(
        [system_message] + state.messages
    )
    print(f"AI message: {ai_message.content}")
    
    # Return the update.
    return {
        "messages": [ai_message]
    }


def act_node(state: ReActState) -> ReActState:
    """The acting node for the ReAct agent."""

    outputs = []
    for tool_call in state.messages[-1].tool_calls:
        tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
        outputs.append(
            ToolMessage(
                content=json.dumps(tool_result),
                name=tool_call["name"],
                tool_call_id=tool_call["id"]
            )
        )
        print(f"Tool call: {tool_call}, result: {tool_result}")
    return {
        "messages": outputs
    }

The `reason_node` is the LLM node. It takes the current state as input and returns the updated state. The LLM will generate a response based on the messages in the state and decide whether to use a tool or provide a final answer.

The `act_node` is the tool node. It takes the current state as input and returns the updated state. If the LLM has decided to use a tool, the tool node will execute the tool and add the result to the messages in the state.

We would assume the agent would oscillate between the two nodes. This is true. But when would we stop? We need a stopping criterion.

In [ ]:
def should_continue(state: ReActState) -> bool:
    """Decide whether to continue or end based on the last message."""

    # Get the last message.
    messages = state.messages
    last_message = messages[-1]

    # If there are tool calls, continue. Otherwise, end.
    if not last_message.tool_calls:
        return END
    else:
        return "act_node"

The method `should_continue` checks the last message in the state. If the last message is a final answer, which is indicated by the absence of tool calls, the method returns `END`, indicating that the workflow should stop. Otherwise, it returns `act_node`. Thus the execution would stop as soon as the LLM stops producing tool calls or if the recursion limit is reached.

In [ ]:
# Create the graph using the state.
graph = StateGraph(ReActState)

# Add the nodes.
graph.add_node(reason_node)
graph.add_node(act_node)

# Add the edges.
graph.add_edge(START, "reason_node")
graph.add_conditional_edges(
    "reason_node",
    should_continue,
    ["act_node", END]
)
graph.add_edge("act_node", "reason_node")

# Create the agent.
react_agent = graph.compile()

# Render the agent.
display(react_agent)

The above code does not contain any big surprises. We define the graph, add the nodes and edges, and create the agent by compiling the graph. The `reason_node` is our start node. We have a conditional edge that checks whether the agent should continue or stop. If it should continue, it goes to the `act_node`, otherwise it ends the workflow. Finally, we add an edge from the `act_node` back to the `reason_node`, allowing the agent to continue reasoning after acting.

The agent is runnable. We can now invoke it with a prompt:

In [ ]:
result = react_agent.invoke(
    {
        "messages": [HumanMessage(content="Generate 4 examples that use the math tool and the sort tool. After that run the tools.")]
    },
    config={"callbacks": [langfuse_handler]}
)
print(result)

# Done.